Lipinski annotations on the 'Drug' Subset from the CSD
The drug subset includes FDA approved-molecules which do not necessarily compy with Lipinski's Ro5

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from upsetplot import UpSet, from_indicators
from venn import venn

from rdkit import Chem
from rdkit.Chem import Descriptors

from ccdc import io

In [ ]:
#extract drug subset
drug_reader = io.EntryReader(subset=io.Subsets.DRUG)

Using csd interator to extract the molecule object for each entry within the drug subset and the SMILES string components. 

In [ ]:
drug_output = Path("all_drug_properties.csv")
# worth making more stable
def csd_iterator(drug_reader)
    for entry in drug_reader:
        try:
            mol = entry.molecule
            smiles = mol.components #changed from mol.smiles
        except RuntimeError: 
            continue
        yield mol.identifier, smiles

Calculate the molecular properties: molecular weight, number of H-bond donors, number of H-bond acceptors, Crippen logP
Using RDKit for each entry iterated through
This will also output failed molecules into a separate json file

In [ ]:
records = []
for i, (identifier, smiles) in enumerate(csd_iterator()): 
    if not smiles: # Iterates over each entry in the database, if there is no SMIlES string, skip.
        continue
    mol = Chem.MolFromSmiles(smiles) # If there is a SMILES string, generate an RDKit molecule object.

    if not mol: # If RDKit cannot generate a molecule object from the SMILES string, skip. 
        continue
    # it might be a good idea to create lists with failed molecules and the errors and dump them into separate json files when iteration is over
    logp_crippen = Descriptors.MolLogP(mol)
    mol_weight = Descriptors.ExactMolWt(mol)
    hb_donors = Descriptors.NumHDonors(mol)
    hb_acceptors = Descriptors.NumHAcceptors(mol)

    record = {
        "identifier": identifier,
        "smiles": smiles, 
        "HBD": hb_donors,
        "HBA": hb_acceptors,
        "logP": logp_crippen,
        "MW": mol_weight,
    }
    records.append(record)

df = pd.DataFrame(records)
df.to_csv(drug_output)

In [ ]:
df.head()

In [ ]:
#could compare stats from separated components vs single SMILES strings for each entry
#Diana's suggestion - separate the components into refcodes

Annotating the entire CSD database and compliancy with Lipinski's Ro5

In [ ]:
#Insert code here for how to calculate and extract molecular properties form the csd and output into a csv file (will not include the extracted entries on github due to IP)

In [ ]:
#find input csv file for entire csd database entries presumably containing the relevant information
csd_df = pd.read_csv("all_drug_properties.csv", index_col = 0)

In [ ]:
lipinski_columns = ["Ro5- HBD", "Ro5- HBA", "Ro5- logP", "Ro5- MW"] # this variable looks unused at this point

thresholds = {"HBD": 5, "HBA": 10, "logP": 5, "MW": 500}

for idx, row in csd_df.iterrows():
    for col, thresh in thresholds.items():
        df.loc[idx, f"{col}_bin"] = row[col] <= thresh

In [ ]:
for col in thresholds:
    df[f"{col}_bin"] = df[f"{col}_bin"].astype('int')

In [ ]:
df.head()

In [ ]:
bin_cols = ["HBD_bin", "HBA_bin", "logP_bin", "MW_bin"]
df["bin_sum"] = df[bin_cols].sum(axis=1)

In [ ]:
df

Output table to show how many of Lipinski's conditions are satisfied for each entry

In [ ]:
row_summary = (df["bin_sum"].value_counts().reindex([4,3,2,1,0], fill_value=0).to_frame(name="count"))
row_summary["percent"] = 100 * row_summary["count"] / row_summary["count"].sum()

row_summary["percent"] = row_summary["percent"].round(2)

row_summary

In [ ]:
row_summary["count"].sum() == len(df)

Number of entries satisfying each Ro5 rule

In [ ]:
col_summary = df[bin_cols].sum().to_frame(name="count")

col_summary["percent"] = 100 * col_summary["count"] / len(csd_df)

col_summary["percent"] = col_summary["percent"].round(2)

col_summary

In [ ]:
sets = {c: set(df.index[df[c] == 1]) for c in bin_cols}
plt.figure(figsize=(8, 8))
# use {size} not {count}; {percentage} is percent of the union
v = venn(sets, fmt="{size}\n({percentage:.1f}%)")
plt.title("Elliptical-style 4-set Venn")
plt.show()

Generate an UpSet plot for combination of properties that satisfy the Ro5 (not necessarily all at once)

In [ ]:
bin_cols = ["HBD_bin", "HBA_bin", "logP_bin", "MW_bin"]

#csd_df[bin_cols] = csd_df[bin_cols].fillna(0).astype(int)
df[bin_cols] = df[bin_cols].fillna(0).astype(int)

# Ensure binary / boolean
#csd_df[bin_cols] = csd_df[bin_cols].fillna(0).astype(bool)
df[bin_cols] = df[bin_cols].fillna(0).astype(bool)

# Create upset data (NO sort_by here)
#data = from_indicators(bin_cols, csd_df)
data = from_indicators(bin_cols, df)

# Plot
plt.figure(figsize=(10, 6))
up = UpSet(
    data,
    subset_size="count",
    show_counts=True,
    sort_by="cardinality"
)

up.plot()
plt.suptitle("UpSet plot for properties", y=1.02)
plt.show()